In [1]:
import sys
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')

from cox import SA
import yaml
import jax
import jax.numpy as jnp
import haiku as hk

In [2]:
config_path = '../config.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

In [3]:
agent = SA(config, 42)

In [4]:
xs = agent.data['seqs']
ts = agent.data['ts']
cs = agent.data['cs']

n, H, _ = xs.shape
# xs = jnp.concatenate((xs, jnp.tile(jnp.eye(H), (n, 1, 1))), axis=-1)
# agent.integrated_brier_score(xs[:,0], ts, cs)

In [5]:
xs[:].shape

(467, 5, 10)

In [14]:
surv = agent.survival_curve(jnp.expand_dims(xs[:,0], axis=1))

In [15]:
surv

Array([[1.        , 0.41933197, 0.17583932, 0.07373506, 0.03091946,
        0.01296552],
       [1.        , 0.49048525, 0.24057578, 0.11799888, 0.0578767 ,
        0.02838767],
       [1.        , 0.5459015 , 0.29800844, 0.16268325, 0.08880902,
        0.04848098],
       ...,
       [1.        , 0.5715708 , 0.32669318, 0.18672827, 0.10672843,
        0.06100285],
       [1.        , 0.5910957 , 0.34939414, 0.20652537, 0.12207626,
        0.07215875],
       [1.        , 0.5540023 , 0.30691856, 0.17003359, 0.094199  ,
        0.05218646]], dtype=float32)

In [7]:
jnp.expand_dims(xs[:,0], axis=1).shape

(467, 1, 10)

In [8]:
xs.shape

(467, 5, 10)

In [9]:
logits = agent.forward(agent.state.params, xs[:,0]).squeeze()
# logits = jnp.expand_dims(logits, axis=1)
log_hs = jax.nn.log_sigmoid(logits)
surv = jnp.exp(jnp.cumsum(log_hs - logits, axis=1))

In [10]:
logits.shape

(467, 5)

In [11]:
logits.shape

(467, 5)

In [14]:
surv = jnp.insert(surv, 0, 1.0, axis=1)

In [17]:
agent.integrated_brier_score(xs[:,0], ts, cs)

Array(0.22350727, dtype=float32)

In [94]:
def forward_fn(x):
    linear = hk.Linear(1)
    alpha_t = hk.Bias()
    out1 = linear(x)
    # out1 = jnp.transpose(out1, (0, 2, 1))
    out = alpha_t(out1)
    # return out1
    # return {'line':out1, 'bias': out}

In [95]:
_some_input = xs[0]
_some_input = _some_input.reshape(1, *_some_input.shape)
_key = jax.random.PRNGKey(0)
forward = hk.without_apply_rng(hk.transform(forward_fn))
params = forward.init(_key, _some_input)
forward = forward.apply

In [96]:
first= jnp.expand_dims(xs[:, 0], axis=1)

In [97]:
first.shape

(467, 1, 10)

In [98]:
out = forward(params, first)

ValueError: 'bias/b' with retrieved shape (5, 1) does not match shape=(1, 1) dtype=dtype('float32')

In [93]:
out + jnp.array([1, 2, 3, 4, 5])

Array([[[0.6796086 , 1.6796086 , 2.6796086 , 3.6796086 , 4.6796083 ]],

       [[1.2660204 , 2.2660205 , 3.2660205 , 4.2660203 , 5.2660203 ]],

       [[0.64161646, 1.6416165 , 2.6416163 , 3.6416163 , 4.6416163 ]],

       ...,

       [[0.6228856 , 1.6228856 , 2.6228857 , 3.6228857 , 4.6228857 ]],

       [[0.5117534 , 1.5117533 , 2.5117533 , 3.5117533 , 4.5117536 ]],

       [[0.56248665, 1.5624866 , 2.5624866 , 3.5624866 , 4.5624866 ]]],      dtype=float32)

In [71]:
h.shape

(467, 1, 5)

In [72]:
out['bias'].shape

(467, 1, 5)

In [73]:
logits = out['bias']
log_hs = jax.nn.log_sigmoid(logits)
surv = jnp.exp(jnp.cumsum(log_hs - logits, axis=1))

In [76]:
xs[:,0].shape

(467, 10)